# coerce-float-arg-to-array — worked example 3: add_forward: coerce either operand before broadcasting with einops

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `coerce-float-arg-to-array`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Binary ops in an autograd engine may receive a Python scalar on **either** side: `add(3.0, t)` or `add(t, 3.0)`. The forward pass must coerce whichever operand is a raw scalar to an array so a single broadcasting rule applies regardless of argument order. Here we coerce both, broadcast a per-batch scalar against a matrix with `einops.repeat`, and add.

## Worked solution

**Step 1 — coerce each operand independently.** A helper `_as_array` wraps a real `int`/`float` (but not `bool`) as `np.array(float(v))` and passes ndarrays through. Applying it to both `a` and `b` means the rest of the function never branches on argument order.

**Step 2 — align shapes with einops.** We have a `(batch,)` vector of per-row offsets and a `(batch, feat)` matrix. `repeat(offset, 'b -> b f', f=feat)` materializes the broadcast explicitly so the addition is unambiguous and the shapes are visible.

**Step 3 — add.** Element-wise `mat + offset_full` now works because both are arrays of the same shape.

**Step 4 — sanity print.** We confirm the output shape and that adding a coerced scalar offset of `0.0` is the identity, while a coerced `2` (int) adds `2.0` everywhere — proving the int was promoted to float math.

**Why it works:** coercing both operands up front collapses the four cases (scalar/array × left/right) into one array-array path, and `einops.repeat` makes the broadcast explicit instead of implicit.

In [ ]:
import numpy as np

def _as_array(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return np.array(float(v))
    return v

def add_with_offset(mat, offset):
    mat = _as_array(mat)
    offset = _as_array(offset)            # scalar -> 0-D array
    b, f = mat.shape
    if offset.ndim == 0:                  # broadcast a single scalar across all rows
        offset = np.full((b,), float(offset))
    offset_full = repeat(offset, 'b -> b f', f=f)
    return mat + offset_full

np.random.seed(0)
mat = np.random.randn(3, 4)
out_scalar = add_with_offset(mat, 2)      # int scalar -> +2.0 everywhere
out_id = add_with_offset(mat, 0.0)        # float 0.0 -> identity
print("shape:", out_scalar.shape)
print("int 2 added everywhere:", np.allclose(out_scalar - mat, 2.0))
print("0.0 is identity:", np.allclose(out_id, mat))